# Day 5B: Causal Inference with Text

This notebook is about causal designs where text enters the research design. Text can be a treatment, an outcome, a confounder, a mediator, a moderator, or a measurement device. Those roles imply different estimands and different risks.

The examples use simulated data because the true data-generating process is known. That makes it easier to see what goes wrong when we use text-derived variables naively.

By the end you should be able to:

1. Distinguish prediction from causal identification.
2. Estimate a treatment effect when text is the treatment.
3. Use pre-treatment text as a high-dimensional confounder proxy.
4. Estimate effects on text-derived outcomes.
5. Understand why controlling for text mediators changes the estimand.
6. Use pre-treatment text to explore treatment-effect heterogeneity.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import roc_auc_score

pd.set_option('display.max_colwidth', 140)
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

## 1. Helpers

These functions are intentionally short. The point is to keep the causal estimators visible rather than hiding them in a package.

In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))


def difference_in_means(df, outcome, treatment):
    treated = df.loc[df[treatment] == 1, outcome]
    control = df.loc[df[treatment] == 0, outcome]
    return treated.mean() - control.mean()


def ols_summary(df, outcome, predictors):
    X = df[predictors].to_numpy(dtype=float)
    y = df[outcome].to_numpy(dtype=float)
    X_design = np.column_stack([np.ones(len(X)), X])
    names = ['intercept'] + predictors

    beta = np.linalg.lstsq(X_design, y, rcond=None)[0]
    residuals = y - X_design @ beta
    dof = max(len(y) - X_design.shape[1], 1)
    sigma2 = (residuals @ residuals) / dof
    cov = sigma2 * np.linalg.pinv(X_design.T @ X_design)
    se = np.sqrt(np.diag(cov))

    return pd.DataFrame({'term': names, 'estimate': beta, 'std_error': se})


def ipw_ate(y, treatment, propensity):
    propensity = np.clip(propensity, 0.05, 0.95)
    treated_part = treatment * y / propensity
    control_part = (1 - treatment) * y / (1 - propensity)
    return np.mean(treated_part - control_part)


def standardized_mean_difference(x, treatment, weights=None):
    x = np.asarray(x)
    treatment = np.asarray(treatment)
    if weights is None:
        weights = np.ones_like(x, dtype=float)
    weights = np.asarray(weights, dtype=float)

    x1 = x[treatment == 1]
    x0 = x[treatment == 0]
    w1 = weights[treatment == 1]
    w0 = weights[treatment == 0]

    m1 = np.average(x1, weights=w1)
    m0 = np.average(x0, weights=w0)
    v1 = np.average((x1 - m1) ** 2, weights=w1)
    v0 = np.average((x0 - m0) ** 2, weights=w0)
    pooled = np.sqrt((v1 + v0) / 2)
    return (m1 - m0) / pooled

In [ ]:
def draw_dag(edges, positions, title='DAG'):
    plt.figure(figsize=(6, 4))
    ax = plt.gca()
    for node, (x, y) in positions.items():
        ax.scatter([x], [y], s=1800, color='white', edgecolor='black', zorder=2)
        ax.text(x, y, node, ha='center', va='center', zorder=3)
    for source, target in edges:
        x1, y1 = positions[source]
        x2, y2 = positions[target]
        ax.annotate('', xy=(x2, y2), xytext=(x1, y1), arrowprops={'arrowstyle': '->', 'lw': 1.5})
    ax.set_title(title)
    ax.axis('off')
    plt.tight_layout()

## 2. Simulating text from latent traits

We simulate short pre-treatment texts. The texts are generated from a latent ideology score, so they contain information about a confounder. This lets us demonstrate text adjustment when the true confounder is partly hidden.

In [ ]:
progressive_words = np.array(['equality', 'welfare', 'union', 'climate', 'rights', 'justice', 'public', 'redistribution'])
conservative_words = np.array(['market', 'tax', 'business', 'border', 'security', 'tradition', 'freedom', 'private'])
neutral_words = np.array(['policy', 'community', 'family', 'country', 'work', 'future', 'people', 'government'])

threat_words = np.array(['danger', 'crime', 'threat', 'crisis', 'fear', 'attack', 'risk', 'emergency'])
rights_words = np.array(['rights', 'dignity', 'fairness', 'freedom', 'equal', 'voice', 'justice', 'respect'])
positive_words = np.array(['good', 'hopeful', 'strong', 'successful', 'beneficial', 'promising'])
negative_words = np.array(['bad', 'harmful', 'weak', 'failed', 'costly', 'damaging'])


def make_pre_text(ideology, length=35):
    p_conservative = sigmoid(1.6 * ideology)
    words = []
    for _ in range(length):
        draw = rng.random()
        if draw < 0.45:
            words.append(rng.choice(conservative_words if rng.random() < p_conservative else progressive_words))
        else:
            words.append(rng.choice(neutral_words))
    return ' '.join(words)


def make_message_text(treatment, length=30):
    frame_words = threat_words if treatment == 1 else rights_words
    words = list(rng.choice(frame_words, size=12, replace=True))
    words += list(rng.choice(neutral_words, size=length - 12, replace=True))
    rng.shuffle(words)
    return ' '.join(words)


def make_outcome_text(latent_score, length=28):
    p_positive = sigmoid(latent_score)
    words = []
    for _ in range(length):
        if rng.random() < 0.50:
            words.append(rng.choice(positive_words if rng.random() < p_positive else negative_words))
        else:
            words.append(rng.choice(neutral_words))
    return ' '.join(words)

## 3. Case A: Text as treatment

In this design, the treatment is a text. For example, participants may be randomly assigned to read a threat-framed message or a rights-framed message.

If assignment is randomized, the difference in outcomes identifies the average treatment effect. If assignment is not randomized, causal identification requires adjustment or a design argument.

### Methodology formulas: causal inference with text

For a binary treatment $T_i$, the average treatment effect is

$$\tau = E[Y_i(1)-Y_i(0)].$$

Identification from observational data requires assumptions such as conditional ignorability and overlap:

$$(Y_i(1),Y_i(0)) \perp T_i \mid X_i, \qquad 0 < e(X_i) < 1,$$

where

$$e(X_i)=P(T_i=1\mid X_i)$$

is the propensity score. An inverse-probability weighted estimator is

$$\hat{\tau}_{IPW}=\frac{1}{N}\sum_{i=1}^N \left[\frac{T_iY_i}{\hat{e}(X_i)} - \frac{(1-T_i)Y_i}{1-\hat{e}(X_i)}\right].$$

Balance diagnostics compare treated and control covariate distributions, often using standardized mean differences:

$$\mathrm{SMD}_j = \frac{\bar{X}_{j,T=1}-\bar{X}_{j,T=0}}{s_j}.$$

When text is an outcome, the causal estimand targets a text-derived function $g(\cdot)$:

$$\tau_g = E[g(\mathrm{Text}_i(1))-g(\mathrm{Text}_i(0))].$$

When text is a mediator $M_i$, a simple decomposition distinguishes the total effect from direct and mediated pathways:

$$TE = E[Y(1,M(1))-Y(0,M(0))],$$

$$NDE = E[Y(1,M(0))-Y(0,M(0))], \qquad NIE = E[Y(1,M(1))-Y(1,M(0))].$$

The notebook uses simulations because they make the hidden potential outcomes, confounding, and measurement error visible.


In [ ]:
edges = [('Message text', 'Outcome')]
positions = {'Message text': (-1, 0), 'Outcome': (1, 0)}
draw_dag(edges, positions, title='Randomized text as treatment')

In [ ]:
n = 2500
randomized = pd.DataFrame({'treatment': rng.binomial(1, 0.5, size=n)})
randomized['message_text'] = randomized['treatment'].apply(make_message_text)
randomized['outcome'] = 1.2 * randomized['treatment'] + rng.normal(0, 1, size=n)

true_ate_randomized = 1.2
estimated_ate_randomized = difference_in_means(randomized, 'outcome', 'treatment')

print('true ATE:', round(true_ate_randomized, 3))
print('difference in means:', round(estimated_ate_randomized, 3))
randomized.head()

In [ ]:
plt.figure(figsize=(6, 4))
sns.violinplot(data=randomized, x='treatment', y='outcome', inner='quartile')
plt.xticks([0, 1], ['rights frame', 'threat frame'])
plt.title('Randomized text treatment: outcome by frame')
plt.xlabel('Message frame')
plt.tight_layout()

## 4. Case B: Pre-treatment text as a confounder proxy

Now assignment is not randomized. People with different latent ideology are more likely to receive different messages, and ideology also affects the outcome. In real research, we may not observe ideology directly, but we may observe pre-treatment text that contains traces of it.

In [ ]:
edges = [('Ideology', 'Message text'), ('Ideology', 'Outcome'), ('Message text', 'Outcome'), ('Pre-treatment text', 'Message text'), ('Ideology', 'Pre-treatment text')]
positions = {
    'Ideology': (0, 1.2),
    'Pre-treatment text': (-1.6, 0.2),
    'Message text': (0, -0.6),
    'Outcome': (1.6, 0.2)
}
draw_dag(edges, positions, title='Text treatment with confounding')

In [ ]:
n = 3000
confounded = pd.DataFrame({
    'ideology': rng.normal(0, 1, size=n),
    'age_std': rng.normal(0, 1, size=n)
})

confounded['pre_text'] = confounded['ideology'].apply(make_pre_text)
confounded['treatment_prob'] = sigmoid(-0.2 + 1.15 * confounded['ideology'] + 0.25 * confounded['age_std'])
confounded['treatment'] = rng.binomial(1, confounded['treatment_prob'])
confounded['message_text'] = confounded['treatment'].apply(make_message_text)

true_ate = 2.0
confounded['outcome'] = true_ate * confounded['treatment'] + 1.4 * confounded['ideology'] + 0.3 * confounded['age_std'] + rng.normal(0, 1, size=n)

confounded[['pre_text', 'treatment', 'outcome', 'ideology']].head()

In [ ]:
naive = difference_in_means(confounded, 'outcome', 'treatment')
age_adjusted = ols_summary(confounded, 'outcome', ['treatment', 'age_std']).query("term == 'treatment'")['estimate'].iloc[0]
oracle = ols_summary(confounded, 'outcome', ['treatment', 'age_std', 'ideology']).query("term == 'treatment'")['estimate'].iloc[0]

pd.DataFrame({
    'estimator': ['true ATE', 'naive difference', 'age adjusted', 'oracle adjusted for ideology'],
    'estimate': [true_ate, naive, age_adjusted, oracle]
})

### High-dimensional text adjustment using propensity scores

We use TF-IDF features from pre-treatment text to predict treatment assignment. This approximates adjustment for the latent confounder because the text was generated from ideology.

This is a teaching example, not a guarantee: text adjustment only helps if the text contains the relevant confounding information and is pre-treatment.

In [ ]:
tfidf = TfidfVectorizer(min_df=5, max_features=1200)
X_text = tfidf.fit_transform(confounded['pre_text'])

propensity_model = LogisticRegression(max_iter=1000)
propensity_model.fit(X_text, confounded['treatment'])
confounded['text_propensity'] = propensity_model.predict_proba(X_text)[:, 1]

text_ipw = ipw_ate(
    confounded['outcome'].to_numpy(),
    confounded['treatment'].to_numpy(),
    confounded['text_propensity'].to_numpy()
)

estimates = pd.DataFrame({
    'estimator': ['true ATE', 'naive difference', 'age adjusted', 'text-propensity IPW', 'oracle adjusted'],
    'estimate': [true_ate, naive, age_adjusted, text_ipw, oracle]
})
estimates

In [ ]:
plt.figure(figsize=(8, 4))
sns.barplot(data=estimates, x='estimate', y='estimator', color='#4C72B0')
plt.axvline(true_ate, color='black', linestyle='--', label='true ATE')
plt.title('Estimating a text-treatment effect under confounding')
plt.xlabel('Estimated treatment effect')
plt.ylabel('')
plt.legend()
plt.tight_layout()

In [ ]:
confounded['ipw_weight'] = np.where(
    confounded['treatment'] == 1,
    1 / np.clip(confounded['text_propensity'], 0.05, 0.95),
    1 / np.clip(1 - confounded['text_propensity'], 0.05, 0.95)
)

balance = pd.DataFrame({
    'covariate': ['ideology', 'age_std'],
    'unweighted_smd': [
        standardized_mean_difference(confounded['ideology'], confounded['treatment']),
        standardized_mean_difference(confounded['age_std'], confounded['treatment'])
    ],
    'text_ipw_smd': [
        standardized_mean_difference(confounded['ideology'], confounded['treatment'], confounded['ipw_weight']),
        standardized_mean_difference(confounded['age_std'], confounded['treatment'], confounded['ipw_weight'])
    ]
})

balance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(data=confounded, x='text_propensity', hue='treatment', bins=30, element='step', stat='density', common_norm=False, ax=axes[0])
axes[0].set_title('Text-estimated treatment propensity')
axes[0].set_xlabel('Estimated P(treatment | pre-treatment text)')

balance_long = balance.melt(id_vars='covariate', var_name='weighting', value_name='standardized_mean_difference')
sns.barplot(data=balance_long, x='standardized_mean_difference', y='covariate', hue='weighting', ax=axes[1])
axes[1].axvline(0, color='black', linewidth=1)
axes[1].set_title('Balance diagnostics')
axes[1].set_xlabel('Standardized mean difference')
axes[1].set_ylabel('')

plt.tight_layout()

### Additional demo: propensity trimming and weight diagnostics

Inverse-probability weights can become unstable when estimated propensities are close to 0 or 1. This cell shows how trimming/clipping changes the effect estimate and effective sample size.

In [ ]:
trim_rows = []
for lower_bound in [0.01, 0.02, 0.05, 0.10, 0.15]:
    clipped = np.clip(confounded['text_propensity'].to_numpy(), lower_bound, 1 - lower_bound)
    weights = np.where(confounded['treatment'] == 1, 1 / clipped, 1 / (1 - clipped))
    estimate = ipw_ate(
        confounded['outcome'].to_numpy(),
        confounded['treatment'].to_numpy(),
        clipped
    )
    effective_n = weights.sum() ** 2 / np.sum(weights ** 2)
    trim_rows.append({
        'propensity_clip': lower_bound,
        'ipw_estimate': estimate,
        'max_weight': weights.max(),
        'effective_sample_size': effective_n
    })

trim_sensitivity = pd.DataFrame(trim_rows)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.lineplot(data=trim_sensitivity, x='propensity_clip', y='ipw_estimate', marker='o', ax=axes[0])
axes[0].axhline(true_ate, color='black', linestyle='--', label='true ATE')
axes[0].set_title('Effect estimate under propensity clipping')
axes[0].set_xlabel('Lower/upper clipping bound')
axes[0].set_ylabel('IPW estimate')
axes[0].legend()

sns.lineplot(data=trim_sensitivity, x='propensity_clip', y='effective_sample_size', marker='o', ax=axes[1])
axes[1].set_title('Effective sample size after weighting')
axes[1].set_xlabel('Lower/upper clipping bound')
axes[1].set_ylabel('Effective sample size')

plt.tight_layout()
trim_sensitivity.round(3)

## 5. Text adjustment with lower-dimensional representations

Another common approach is to reduce high-dimensional text to a small number of components and include those components as controls. This is a design choice, not a magic fix.

In [ ]:
max_components = max(1, min(X_text.shape[1] - 1, X_text.shape[0] - 1))
component_grid = sorted({0, 2, 5, 10, min(20, max_components), max_components})

component_results = []
for n_components in component_grid:
    if n_components == 0:
        estimate = age_adjusted
    else:
        svd = TruncatedSVD(n_components=n_components, random_state=RANDOM_SEED)
        components = svd.fit_transform(X_text)
        component_cols = [f'text_component_{i}' for i in range(n_components)]
        component_df = pd.DataFrame(components, columns=component_cols, index=confounded.index)
        tmp = pd.concat([confounded[['outcome', 'treatment', 'age_std']], component_df], axis=1)
        estimate = ols_summary(tmp, 'outcome', ['treatment', 'age_std'] + component_cols).query("term == 'treatment'")['estimate'].iloc[0]
    component_results.append({'text_components': n_components, 'estimated_effect': estimate})

component_results = pd.DataFrame(component_results)
component_results

In [ ]:
plt.figure(figsize=(6, 4))
sns.lineplot(data=component_results, x='text_components', y='estimated_effect', marker='o')
plt.axhline(true_ate, color='black', linestyle='--', label='true ATE')
plt.title('Text controls via TF-IDF/SVD components')
plt.xlabel('Number of text components included')
plt.ylabel('Estimated treatment effect')
plt.legend()
plt.tight_layout()

## 6. Case C: Text as outcome

Sometimes the outcome is text: speeches, open-ended survey responses, posts, comments, or manifestos. Then the causal estimand is an effect on a text-derived measure.

Measurement is now part of the causal estimand. A noisy dictionary or classifier can attenuate or distort the estimated effect.

In [ ]:
n = 2500
text_outcome = pd.DataFrame({'treatment': rng.binomial(1, 0.5, size=n)})

true_effect_on_latent_text = 1.0
text_outcome['latent_positive_tone'] = true_effect_on_latent_text * text_outcome['treatment'] + rng.normal(0, 1, size=n)
text_outcome['response_text'] = text_outcome['latent_positive_tone'].apply(make_outcome_text)

def dictionary_tone(text):
    tokens = str(text).split()
    return sum(token in positive_words for token in tokens) - sum(token in negative_words for token in tokens)

text_outcome['measured_tone'] = text_outcome['response_text'].apply(dictionary_tone)

latent_effect = difference_in_means(text_outcome, 'latent_positive_tone', 'treatment')
measured_effect = difference_in_means(text_outcome, 'measured_tone', 'treatment')

pd.DataFrame({
    'outcome': ['latent positive tone', 'dictionary-measured tone'],
    'estimated_effect': [latent_effect, measured_effect]
})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.kdeplot(data=text_outcome, x='latent_positive_tone', hue='treatment', common_norm=False, ax=axes[0])
axes[0].set_title('Latent text outcome')

sns.histplot(data=text_outcome, x='measured_tone', hue='treatment', bins=20, element='step', stat='density', common_norm=False, ax=axes[1])
axes[1].set_title('Dictionary-measured text outcome')

plt.tight_layout()

## 7. Case D: Text as mediator

A mediator lies on the causal path. If the treatment changes text, and the text changes the outcome, controlling for the text blocks part of the effect. That estimates a direct effect, not the total effect.

In [ ]:
edges = [('Treatment', 'Mediator text'), ('Mediator text', 'Outcome'), ('Treatment', 'Outcome')]
positions = {'Treatment': (-1.4, 0), 'Mediator text': (0, 1), 'Outcome': (1.4, 0)}
draw_dag(edges, positions, title='Text as mediator')

In [ ]:
n = 3000
mediation = pd.DataFrame({'treatment': rng.binomial(1, 0.5, size=n)})
mediation['latent_mediator'] = 0.9 * mediation['treatment'] + rng.normal(0, 1, size=n)
mediation['mediator_text'] = mediation['latent_mediator'].apply(make_outcome_text)
mediation['measured_mediator'] = mediation['mediator_text'].apply(dictionary_tone)

direct_effect = 0.8
mediator_to_outcome = 1.1
mediation['outcome'] = direct_effect * mediation['treatment'] + mediator_to_outcome * mediation['latent_mediator'] + rng.normal(0, 1, size=n)

total_model = ols_summary(mediation, 'outcome', ['treatment'])
direct_oracle_model = ols_summary(mediation, 'outcome', ['treatment', 'latent_mediator'])
direct_measured_model = ols_summary(mediation, 'outcome', ['treatment', 'measured_mediator'])

mediation_estimates = pd.DataFrame({
    'model': ['total effect model', 'direct effect with true mediator', 'direct-ish effect with measured text mediator'],
    'treatment_estimate': [
        total_model.query("term == 'treatment'")['estimate'].iloc[0],
        direct_oracle_model.query("term == 'treatment'")['estimate'].iloc[0],
        direct_measured_model.query("term == 'treatment'")['estimate'].iloc[0]
    ]
})

mediation_estimates

In [ ]:
plt.figure(figsize=(7, 4))
sns.barplot(data=mediation_estimates, x='treatment_estimate', y='model', color='#4C72B0')
plt.axvline(direct_effect, color='black', linestyle='--', label='true direct effect')
plt.title('Controlling for a text mediator changes the estimand')
plt.xlabel('Treatment coefficient')
plt.ylabel('')
plt.legend()
plt.tight_layout()

## 8. Case E: Text as moderator for heterogeneous effects

Pre-treatment text can define subgroups. Here treatment effects are larger among respondents whose pre-treatment text reveals a more conservative latent ideology. In real applications, this requires strong assumptions and careful validation.

In [ ]:
heterogeneity = confounded.copy()
heterogeneity['true_effect'] = 1.0 + 1.2 * (heterogeneity['ideology'] > 0).astype(float)
heterogeneity['outcome_het'] = heterogeneity['true_effect'] * heterogeneity['treatment'] + 1.0 * heterogeneity['ideology'] + rng.normal(0, 1, size=len(heterogeneity))

def conservative_text_score(text):
    tokens = str(text).split()
    return sum(token in conservative_words for token in tokens) - sum(token in progressive_words for token in tokens)

heterogeneity['text_ideology_score'] = heterogeneity['pre_text'].apply(conservative_text_score)
heterogeneity['text_score_group'] = pd.qcut(heterogeneity['text_ideology_score'], q=3, labels=['low', 'middle', 'high'])

group_effects = []
for group, frame in heterogeneity.groupby('text_score_group', observed=True):
    estimate = ols_summary(frame, 'outcome_het', ['treatment', 'age_std']).query("term == 'treatment'")['estimate'].iloc[0]
    group_effects.append({
        'text_score_group': group,
        'estimated_treatment_effect': estimate,
        'mean_true_effect': frame['true_effect'].mean(),
        'n': len(frame)
    })

group_effects = pd.DataFrame(group_effects)
group_effects

In [ ]:
group_plot = group_effects.melt(
    id_vars=['text_score_group'],
    value_vars=['estimated_treatment_effect', 'mean_true_effect'],
    var_name='quantity',
    value_name='effect'
)

plt.figure(figsize=(7, 4))
sns.pointplot(data=group_plot, x='text_score_group', y='effect', hue='quantity', dodge=True)
plt.title('Treatment-effect heterogeneity by pre-treatment text')
plt.xlabel('Pre-treatment text score group')
plt.ylabel('Treatment effect')
plt.tight_layout()

## 9. Design checklist: causal inference with text

Before using text in a causal design, answer these questions:

1. What is the causal estimand?
2. What role does text play: treatment, outcome, confounder, mediator, moderator, or object of study?
3. Is the text pre-treatment or post-treatment?
4. Is the text-derived measure valid for the construct?
5. What confounders remain unmeasured?
6. Could text adjustment block a mediator or open a collider?
7. Are text features stable across domains, time, speakers, or platforms?
8. What simple estimator is the baseline?
9. What sensitivity analysis would change your conclusion?

In [ ]:
checklist = pd.DataFrame({
    'role_of_text': ['treatment', 'outcome', 'confounder proxy', 'mediator', 'moderator'],
    'key_question': [
        'Was exposure to the text randomized or otherwise identified?',
        'Does the text measure validly capture the outcome construct?',
        'Is the text pre-treatment and informative about confounding?',
        'Do we want total effect or direct effect?',
        'Are subgroup definitions pre-treatment and validated?'
    ],
    'main_risk': [
        'Non-random assignment to text',
        'Measurement error in text-derived outcome',
        'High-dimensional adjustment without identification',
        'Controlling away part of the causal effect',
        'Data-driven subgroup discovery and overinterpretation'
    ]
})

checklist

## 10. Student task

Pick one of the five cases above and adapt it to your own project:

- Draw the DAG.
- State the estimand.
- Identify which text is pre-treatment and which text is post-treatment.
- Choose a text representation or measurement model.
- Name the identifying assumption that cannot be tested from the text alone.
- Propose one sensitivity analysis.